# Topic Modeling sobre el corpus desclasificado del 23F

**Caso de uso del TFM** — Análisis de la estructura temática latente de la documentación desclasificada del golpe de Estado del 23 de febrero de 1981, con segmentación por ministerio, fase histórica y nivel de clasificación.

El notebook sigue la plantilla del TFM (planteamiento, desarrollo, exploración de variables, feature engineering, métricas, estudio de patrones, interpretación, errores, aplicación práctica, conclusiones y mejoras) y está pensado para ejecutarse end-to-end con semilla fija para resultados reproducibles.

## Requisitos

```bash
pip install bertopic sentence-transformers umap-learn hdbscan \
            scikit-learn pandas numpy matplotlib seaborn nltk
python -m nltk.downloader stopwords
```

El primer arranque descargará el modelo de embeddings multilingüe (~470 MB).

## 1. Setup y carga del dataset

In [ ]:
# === Imports ===
import os
import re
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import chi2_contingency

import nltk
try:
    from nltk.corpus import stopwords
    STOPWORDS_ES = set(stopwords.words('spanish'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOPWORDS_ES = set(stopwords.words('spanish'))

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

# === Semilla para reproducibilidad ===
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Imports OK')

In [ ]:
# === Carga del dataset ===
CANDIDATE_PATHS = [
    r'C:/Users/migue/OneDrive/Escritorio/Machine-Learning_23F/Dataset_23F.csv',
    'dataset_23f_master.csv',
    'data/dataset_23f_master.csv',
    'data/processed/dataset_23f_master.csv',
    '/mnt/user-data/uploads/dataset_23f_master.csv',
]
DATASET_PATH = next((p for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        f'No se encontro el dataset. Rutas probadas: {CANDIDATE_PATHS}'
    )

df = pd.read_csv(DATASET_PATH, encoding='latin-1')
print(f'Dataset cargado: {DATASET_PATH}')
print(f'Documentos: {len(df)}')
print(f'Columnas: {list(df.columns)}')

In [ ]:
# === Vistazo a las variables de segmentación ===
print('=== Distribución por ministerio ===')
print(df['ministerio'].value_counts(dropna=False))
print('\n=== Distribución por fase ===')
print(df['fase'].value_counts().sort_index())
print('\n=== Distribución por clasificación ===')
print(df['clasificacion'].value_counts(dropna=False))
print('\n=== Longitud de textos (palabras) ===')
print(df['n_palabras'].describe().round(0))

## 2. Preprocesamiento y exploración de variables

**Decisiones de diseño:**

- El embedding semántico se calcula sobre el **texto original mínimamente limpiado** (sin stopwords, sin stemming): los modelos de sentence-transformers están entrenados con texto natural y degradan si se les pasa texto preprocesado al estilo bag-of-words.
- La **eliminación de stopwords se aplica únicamente al vectorizador c-TF-IDF** que genera las etiquetas de los temas, donde sí mejora la interpretabilidad.
- Se incluyen como stopwords adicionales términos administrativos/protocolarios que aparecen masivamente en todos los documentos y no aportan distinción temática (señor, excelentísimo, etc.).

In [ ]:
# === Limpieza ligera del texto para el embedding ===
def limpiar_para_embedding(texto):
    """Limpieza mínima preservando estructura natural del texto."""
    if not isinstance(texto, str):
        return ''
    # Eliminar números de página y artefactos de OCR comunes
    texto = re.sub(r'\n\s*\d+\s*\n', '\n', texto)
    texto = re.sub(r'-{3,}', '', texto)
    # Normalizar espacios en blanco
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['texto_limpio'] = df['texto_completo'].apply(limpiar_para_embedding)

# Filtramos documentos demasiado cortos (no aportan a un tema bien definido)
MIN_PALABRAS = 50
mask_validos = df['n_palabras'] >= MIN_PALABRAS
print(f'Documentos válidos: {mask_validos.sum()} / {len(df)}')
print(f'Descartados por longitud < {MIN_PALABRAS} palabras: {(~mask_validos).sum()}')

df_topic = df[mask_validos].reset_index(drop=True).copy()

In [ ]:
# === Stopwords ampliadas para el vectorizador c-TF-IDF ===
# Términos administrativos/protocolarios sin valor temático
STOPWORDS_EXTRA = {
    # Tratamientos y protocolo
    'señor', 'señores', 'señora', 'excelentísimo', 'excmo', 'don', 'doña',
    'ilustrísimo', 'ilmo', 'usia', 'vd', 'vs',
    # Estructura documental
    'documento', 'documentos', 'asunto', 'referencia', 'anexo', 'adjunto',
    'folio', 'folios', 'página', 'páginas', 'expediente', 'oficio',
    # Verbos protocolarios muy frecuentes
    'tener', 'haber', 'ser', 'estar', 'hacer', 'decir', 'poder', 'deber',
    # Fechas y meses (no aportan tema)
    'enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
    'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre',
    # Lugares y entidades muy frecuentes que dominan todos los temas
    'madrid', 'españa', 'español', 'española', 'españoles',
    # Números escritos
    'uno', 'dos', 'tres', 'cuatro', 'cinco', 'primer', 'primera',
}
STOPWORDS_FINALES = list(STOPWORDS_ES | STOPWORDS_EXTRA)
print(f'Total stopwords: {len(STOPWORDS_FINALES)}')

## 3. Desarrollo: pipeline de BERTopic

El pipeline encadena cinco etapas:

1. **Embedding** con `paraphrase-multilingual-MiniLM-L12-v2`: modelo multilingüe que captura semántica en castellano.
2. **Reducción de dimensionalidad** con UMAP (de 384 dimensiones a 5): conserva estructura local.
3. **Clustering** con HDBSCAN: detecta densidades sin imponer número de clusters; los outliers quedan como tema `-1`.
4. **Tokenización** con `CountVectorizer` y stopwords ampliadas: prepara el vocabulario para c-TF-IDF.
5. **Representación c-TF-IDF**: extrae términos discriminativos por tema (clase) en lugar de por documento.

In [ ]:
# === Embedding model ===
print('Cargando modelo de embeddings...')
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f'Dimensión del embedding: {embedding_model.get_sentence_embedding_dimension()}')

In [ ]:
# === Cálculo de embeddings (puede tardar 1-2 minutos la primera vez) ===
print('Calculando embeddings de los documentos...')
embeddings = embedding_model.encode(
    df_topic['texto_limpio'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f'Shape de embeddings: {embeddings.shape}')

In [ ]:
# === Configuración de UMAP y HDBSCAN ===
# Con n=167 documentos, parámetros conservadores para evitar fragmentación excesiva

umap_model = UMAP(
    n_neighbors=10,        # vecindario moderado: corpus pequeño
    n_components=5,        # dim final reducida típica de BERTopic
    min_dist=0.0,          # clusters compactos
    metric='cosine',
    random_state=RANDOM_SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=8,    # ~5% del corpus: temas con sustancia mínima
    min_samples=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS_FINALES,
    ngram_range=(1, 2),    # incluye bigramas (p. ej. "guardia civil")
    min_df=2,              # término en al menos 2 documentos
    max_df=0.85            # excluir términos en >85% de docs (genéricos)
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
print('Componentes del pipeline configurados')

In [ ]:
# === Entrenamiento del modelo BERTopic ===
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language='spanish',
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(
    df_topic['texto_limpio'].tolist(),
    embeddings=embeddings
)

df_topic['topic'] = topics
print(f'\nTemas detectados: {len(set(topics)) - (1 if -1 in topics else 0)}')
print(f'Outliers (tema -1): {sum(1 for t in topics if t == -1)}')

## 4. Feature engineering: etiquetado de temas

BERTopic asigna un identificador numérico a cada tema, pero la interpretación exige etiquetas legibles. Se generan tres niveles de descripción:

- **Etiqueta automática**: los 4 términos top de c-TF-IDF.
- **Documento representativo**: el documento más cercano al centroide de cada tema, para validar manualmente.
- **Etiqueta manual** (opcional): etiqueta interpretativa que el investigador asigna tras revisar los términos y el documento representativo.

In [ ]:
# === Información de los temas ===
topic_info = topic_model.get_topic_info()
print(topic_info[['Topic', 'Count', 'Name']].to_string(index=False))

In [ ]:
# === Términos top y documento representativo de cada tema ===
def describir_temas(topic_model, df_topic, n_terms=8):
    filas = []
    for topic_id in sorted(set(topic_model.topics_)):
        if topic_id == -1:
            continue
        terms = [w for w, _ in topic_model.get_topic(topic_id)[:n_terms]]
        docs_tema = df_topic[df_topic['topic'] == topic_id]
        n_docs = len(docs_tema)
        # Documento representativo (el más cercano al centroide del tema)
        rep_docs = topic_model.get_representative_docs(topic_id) or []
        rep_titulo = ''
        if rep_docs:
            primer_doc = rep_docs[0][:200]
            match = df_topic[df_topic['texto_limpio'].str.startswith(primer_doc[:80], na=False)]
            if len(match) > 0 and 'titulo' in match.columns:
                rep_titulo = match.iloc[0]['titulo']
        filas.append({
            'topic': topic_id,
            'n_docs': n_docs,
            'terminos': ', '.join(terms),
            'doc_representativo': rep_titulo[:120] if rep_titulo else '(sin título)'
        })
    return pd.DataFrame(filas)

desc_temas = describir_temas(topic_model, df_topic)
print(desc_temas.to_string(index=False))

In [ ]:
# === Etiquetas manuales sugeridas (a revisar tras inspección) ===
# Plantilla: tras leer la tabla anterior, asigna a cada topic_id una etiqueta corta
# Ejecuta y luego edita el diccionario antes del análisis por segmentos
etiquetas_manuales = {
    # 0: 'Operaciones militares',
    # 1: 'Procedimiento judicial',
    # 2: 'Relaciones diplomáticas',
    # ...
}
df_topic['topic_label'] = df_topic['topic'].map(
    lambda t: etiquetas_manuales.get(t, f'Tema {t}' if t >= 0 else 'Sin tema')
)
print('Etiquetas asignadas:', df_topic['topic_label'].value_counts().to_dict())

## 5. Métricas de evaluación

Se evalúan tres dimensiones complementarias:

- **Coherencia c_v** de los términos top por tema: mide hasta qué punto los términos co-ocurren en el corpus.
- **Diversidad léxica**: proporción de términos únicos entre los top-10 de todos los temas. Una diversidad baja indica solapamiento (temas redundantes).
- **Tasa de outliers**: porcentaje de documentos sin asignar (tema `-1`). Una tasa alta sugiere fragmentación o parámetros HDBSCAN demasiado restrictivos.

In [ ]:
# === Coherencia (c_v) ===
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

def coherencia_cv(topic_model, df_topic, top_n=10):
    textos_tokenizados = [doc.lower().split() for doc in df_topic['texto_limpio']]
    diccionario = Dictionary(textos_tokenizados)
    topic_words = []
    for t in sorted(set(topic_model.topics_)):
        if t == -1: continue
        terms = [w for w, _ in topic_model.get_topic(t)[:top_n]]
        if terms:
            topic_words.append(terms)
    if not topic_words:
        return None
    cm = CoherenceModel(
        topics=topic_words,
        texts=textos_tokenizados,
        dictionary=diccionario,
        coherence='c_v'
    )
    return cm.get_coherence()

try:
    cv_score = coherencia_cv(topic_model, df_topic)
    print(f'Coherencia c_v: {cv_score:.3f}')
    print('(Referencia: 0.4–0.55 aceptable, >0.55 buena, >0.7 excelente)')
except Exception as e:
    print(f'No se pudo calcular c_v (gensim no instalado): {e}')
    print('Si quieres c_v: pip install gensim')

In [ ]:
# === Diversidad léxica ===
def diversidad_lexica(topic_model, top_n=10):
    all_terms = []
    for t in sorted(set(topic_model.topics_)):
        if t == -1: continue
        all_terms.extend([w for w, _ in topic_model.get_topic(t)[:top_n]])
    if not all_terms:
        return None
    return len(set(all_terms)) / len(all_terms)

div = diversidad_lexica(topic_model)
print(f'Diversidad léxica: {div:.3f}')
print('(1.0 = todos los términos únicos; <0.7 indica solapamiento entre temas)')

In [ ]:
# === Tasa de outliers ===
n_outliers = sum(1 for t in df_topic['topic'] if t == -1)
tasa = n_outliers / len(df_topic) * 100
print(f'Outliers: {n_outliers} / {len(df_topic)} ({tasa:.1f}%)')
print('(Referencia: <20% es razonable con HDBSCAN; >40% sugiere parámetros muy estrictos)')

## 6. Estudio de patrones: cruces con metadatos

Tres tests de hipótesis para validar las H1, H2 y H3 declaradas en el planteamiento:

- **H1**: distribución de temas por ministerio → chi-cuadrado.
- **H2**: distribución de temas por fase → chi-cuadrado.
- **H3**: distribución de temas por clasificación → chi-cuadrado.

Para cada cruce, además del test estadístico, se genera un heatmap normalizado para inspección visual.

In [ ]:
def cruce_tema_variable(df_topic, variable, etiqueta_var):
    """Tabla de contingencia, chi-cuadrado y heatmap."""
    df_sin_outliers = df_topic[df_topic['topic'] != -1].copy()
    if df_sin_outliers[variable].isna().any():
        df_sin_outliers = df_sin_outliers.dropna(subset=[variable])
    
    ct = pd.crosstab(df_sin_outliers['topic_label'], df_sin_outliers[variable])
    
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    cramer_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1))) if min(ct.shape) > 1 else 0
    
    print(f'\n=== Tema × {etiqueta_var} ===')
    print(f'  chi² = {chi2:.2f}, dof = {dof}, p = {p:.4f}')
    print(f'  V de Cramér = {cramer_v:.3f} (0.1 débil, 0.3 moderada, 0.5 fuerte)')
    
    # Heatmap normalizado por columna (% de cada variable que va a cada tema)
    ct_norm = ct.div(ct.sum(axis=0), axis=1) * 100
    fig, ax = plt.subplots(figsize=(max(6, ct.shape[1]*1.5), max(4, ct.shape[0]*0.5)))
    sns.heatmap(ct_norm, annot=True, fmt='.0f', cmap='Blues', cbar_kws={'label': '% dentro de columna'}, ax=ax)
    ax.set_title(f'Distribución de temas por {etiqueta_var} (% por columna)')
    plt.tight_layout()
    plt.show()
    
    return ct, chi2, p, cramer_v

# H1: por ministerio
ct_min, chi2_min, p_min, v_min = cruce_tema_variable(df_topic, 'ministerio', 'ministerio')

In [ ]:
# H2: por fase histórica
ct_fase, chi2_fase, p_fase, v_fase = cruce_tema_variable(df_topic, 'fase', 'fase histórica')

In [ ]:
# H3: por clasificación
ct_clas, chi2_clas, p_clas, v_clas = cruce_tema_variable(df_topic, 'clasificacion', 'clasificación')

## 7. Visualizaciones del modelo

BERTopic ofrece visualizaciones nativas que ayudan a la interpretación.

In [ ]:
# Mapa 2D de temas (proximidad semántica)
fig = topic_model.visualize_topics()
fig.show()

In [ ]:
# Términos discriminativos por tema (barras)
fig = topic_model.visualize_barchart(top_n_topics=min(10, len(set(df_topic['topic']))-1), n_words=8)
fig.show()

In [ ]:
# Mapa de similitud entre temas
fig = topic_model.visualize_heatmap()
fig.show()

## 8. Análisis de errores

Inspeccionamos los documentos clasificados como outlier (`topic = -1`) para diagnosticar por qué no encajan en ningún tema.

In [ ]:
outliers = df_topic[df_topic['topic'] == -1]
print(f'Documentos outlier: {len(outliers)}')
if len(outliers) > 0 and 'titulo' in outliers.columns:
    print('\nTítulos de los primeros 10 outliers:')
    for t in outliers['titulo'].head(10):
        print(f'  - {t[:100]}')
print('\nLongitud media (palabras) outliers vs resto:')
print(f'  Outliers: {outliers["n_palabras"].mean():.0f}')
print(f'  Resto:    {df_topic[df_topic["topic"] != -1]["n_palabras"].mean():.0f}')

## 9. Robustez: estabilidad del modelo

Repetimos el entrenamiento con tres semillas distintas para evaluar si los temas son estables o si dependen de la inicialización aleatoria de UMAP.

In [ ]:
def comparar_semillas(textos, embeddings, semillas=[42, 123, 999]):
    resultados = []
    for seed in semillas:
        umap_s = UMAP(n_neighbors=10, n_components=5, min_dist=0.0,
                      metric='cosine', random_state=seed)
        hdb_s = HDBSCAN(min_cluster_size=8, min_samples=3,
                        cluster_selection_method='eom', prediction_data=True)
        tm = BERTopic(
            embedding_model=embedding_model, umap_model=umap_s,
            hdbscan_model=hdb_s, vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model, verbose=False
        )
        t, _ = tm.fit_transform(textos, embeddings=embeddings)
        n_temas = len(set(t)) - (1 if -1 in t else 0)
        n_out = sum(1 for x in t if x == -1)
        resultados.append({'seed': seed, 'n_temas': n_temas, 'n_outliers': n_out})
    return pd.DataFrame(resultados)

rob = comparar_semillas(df_topic['texto_limpio'].tolist(), embeddings)
print(rob.to_string(index=False))

## 10. Interpretación, conclusiones y mejoras

Esta sección se completa de forma narrativa tras revisar los resultados anteriores. La plantilla que debe seguir el TFM:

**Interpretación de resultados**
- ¿Qué temas emergieron y qué historia cuenta cada uno?
- ¿Cómo se relacionan con la estructura del acontecimiento (operativa, judicial, diplomática)?

**Validación de hipótesis**
- H1 (ministerio): ¿confirmada por el chi-cuadrado? ¿qué intensidad (V de Cramér)?
- H2 (fase): ¿hay evolución temática clara?
- H3 (clasificación): ¿algún tema sobrerrepresentado en documentos clasificados?

**Aplicación práctica**
- El pipeline es transferible a cualquier corpus institucional con metadatos administrativos y temporales.
- Sirve, por ejemplo, para análisis de archivos diplomáticos desclasificados, sumarios judiciales o corpus parlamentarios.

**Errores y limitaciones**
- Sensibilidad a HDBSCAN y a la semilla de UMAP (verificada en el bloque de robustez).
- Outliers que reflejan documentos atípicos o demasiado breves.
- El c-TF-IDF prioriza términos discriminativos pero puede destacar artefactos de OCR si pasan el filtrado de stopwords.

**Mejoras y próximos pasos**
- Probar con un modelo de embeddings entrenado específicamente en español jurídico-administrativo.
- Aplicar **Dynamic Topic Modeling** sobre la variable fase para modelar explícitamente la evolución temporal.
- Incorporar **guided topic modeling** con seed words por ministerio para evaluar si refuerza H1.
- Comparar con LDA clásico (Mallet) como contraprueba metodológica.